# cld-pi: the `pi` API from Jupyter

Each `%%js` cell runs inside the running claude-p's node sidecar (the global `pi`, the same one
`chrome://inspect` / F12 shows), over CDP on **127.0.0.1:9229**. `%%js 9224` targets another port.

* Start `claude-p` (interactive) first; its sidecar opens the inspector.
* Cells use REPL semantics: top-level `await`, and `let` / `const` may be redeclared.
* A cell that `await`s a dialog finishes when you answer it in the terminal (Esc → `null`).
* Things registered from here (commands, key hooks, seds, …) aren't owned by a plugin, so the
  ◆ list can't turn them off: their disposers go into `nb`, and the last cell runs them.

The spec is [goal-v1.ts](./goal-v1.ts); roughly 60% of the examples below are dialog layouts.

In [ ]:
# %%js [port] [timeout]: evaluate the cell in the claude-p sidecar, where the
# global `pi` lives, over CDP (default port 9229; agent tests use 9224).
# Stdlib only: a minimal WebSocket client, so the kernel needs no packages.
import base64, json, os, socket, struct, urllib.request
from urllib.parse import urlparse

def _ws_connect(url, timeout):
    u = urlparse(url)
    s = socket.create_connection((u.hostname, u.port or 80), timeout=timeout)
    key = base64.b64encode(os.urandom(16)).decode()
    s.sendall((f'GET {u.path} HTTP/1.1\r\nHost: {u.hostname}:{u.port}\r\nUpgrade: websocket\r\n'
               f'Connection: Upgrade\r\nSec-WebSocket-Key: {key}\r\nSec-WebSocket-Version: 13\r\n\r\n').encode())
    head = b''
    while b'\r\n\r\n' not in head:
        chunk = s.recv(4096)
        if not chunk:
            raise ConnectionError('inspector closed the connection')
        head += chunk
    if b' 101 ' not in head.split(b'\r\n', 1)[0]:
        raise ConnectionError(head.split(b'\r\n', 1)[0].decode())
    return s, head.split(b'\r\n\r\n', 1)[1]

def _ws_send(s, text):
    data = text.encode()
    n = len(data)
    hdr = bytes([0x81]) + (bytes([0x80 | n]) if n < 126 else
                           bytes([0x80 | 126]) + struct.pack('>H', n) if n < 65536 else
                           bytes([0x80 | 127]) + struct.pack('>Q', n))
    mask = os.urandom(4)
    s.sendall(hdr + mask + bytes(b ^ mask[i % 4] for i, b in enumerate(data)))

class _Reader:
    def __init__(self, s, buf):
        self.s, self.buf = s, buf
    def take(self, n):
        while len(self.buf) < n:
            chunk = self.s.recv(65536)
            if not chunk:
                raise ConnectionError('inspector closed the connection')
            self.buf += chunk
        out, self.buf = self.buf[:n], self.buf[n:]
        return out
    def message(self):
        parts = []
        while True:
            b0, b1 = self.take(2)
            n = b1 & 0x7f
            if n == 126: n = struct.unpack('>H', self.take(2))[0]
            elif n == 127: n = struct.unpack('>Q', self.take(8))[0]
            payload = self.take(n)
            op = b0 & 0x0f
            if op == 0x8: raise ConnectionError('inspector closed the connection')
            if op in (0x9, 0xA): continue  # ping / pong
            parts.append(payload)
            if b0 & 0x80: return b''.join(parts).decode()

def cdp_eval(code, port=9229, timeout=300):
    """Evaluate `code` in the sidecar (top-level await allowed); return its value."""
    targets = json.load(urllib.request.urlopen(f'http://127.0.0.1:{port}/json/list', timeout=5))
    s, rest = _ws_connect(targets[0]['webSocketDebuggerUrl'], timeout)
    try:
        _ws_send(s, json.dumps({'id': 1, 'method': 'Runtime.evaluate', 'params': {
            'expression': code, 'replMode': True, 'awaitPromise': True, 'returnByValue': True}}))
        r = _Reader(s, rest)
        while True:
            msg = json.loads(r.message())
            if msg.get('id') == 1:
                break
    finally:
        s.close()
    res = msg.get('result', {})
    if 'exceptionDetails' in res:
        d = res['exceptionDetails']
        raise RuntimeError((d.get('exception') or {}).get('description') or d.get('text'))
    v = res.get('result', {})
    return v.get('value', v.get('description', v.get('type')))

try:
    from IPython.core.magic import register_cell_magic

    @register_cell_magic
    def js(line, cell):
        args = line.split()
        return cdp_eval(cell, int(args[0]) if args else 9229, float(args[1]) if len(args) > 1 else 300)
except ImportError:
    pass  # plain Python: call cdp_eval(code) directly

In [ ]:
%%js
globalThis.nb ??= []; // disposers of what these cells register
({ pi: typeof pi, focused: pi.tui.state.focused, size: `${pi.tui.state.cols}x${pi.tui.state.rows}`, arg: { ...pi.arg }, ext: Object.keys(pi.ext) })

## 1. Dialogs

`pi.dialog({ title, body, footer, ...buttons })`: every function-valued key is a button (no
arguments). Returning a string relabels that button and keeps the dialog open; anything else closes
it. The promise resolves with the pressed label, or `null` on Esc. Layout: `eachRow` (`[3, 3]` →
rows of 3, justified; `'vertical'` → a list), `hints` (footer text per focused button), and
`{ timeout, or }` for a non-modal toast. The returned promise is also a handle: `.relabel(from, to)`,
`.text({ title, body, footer })`.

### 1.1 OK / Cancel

In [ ]:
%%js
await pi.dialog({ title: 'Hello', body: 'from Jupyter', OK() {}, Cancel() {} })

### 1.2 A grid that stays open: return the label to keep going (`eachRow: [3, 3, 3, 3]`)

In [ ]:
%%js
let expr = ''
const keys = ['7', '8', '9', '4', '5', '6', '1', '2', '3', '0', '+', '=']
const calc = pi.dialog({
  title: 'add', body: ' ', footer: '= closes', eachRow: [3, 3, 3, 3],
  ...Object.fromEntries(keys.map(k => [k, () => {
    if (k === '=') return // void: close
    expr += k
    calc.text({ body: expr })
    return k // a string: stay open
  }])),
})
await calc
expr.split('+').reduce((sum, n) => sum + Number(n || 0), 0)

### 1.3 A vertical list built from data (`pi.sh` + `eachRow: 'vertical'`)

In [ ]:
%%js
// ${…} values are shell-quoted: the %(…) format needs no manual escaping.
const { stdout } = await pi.sh`git -C ${process.cwd()} branch --format=${'%(refname:short)'}`
const branches = stdout.split('\n').filter(Boolean).slice(0, 8)
const branch = await pi.dialog({
  title: 'branches', footer: 'Enter puts the name in the input box', eachRow: 'vertical',
  ...Object.fromEntries(branches.map(b => [b, () => {}])),
})
if (branch) pi.BUFFER = branch
branch

### 1.4 Hints follow the focus

In [ ]:
%%js
await pi.dialog({
  title: 'Pick a size', footer: '←/→ to compare',
  hints: { Small: 'fast and cheap', Medium: 'the balanced default', Large: 'slow and thorough' },
  Small() {}, Medium() {}, Large() {},
})

### 1.5 Radio row: relabel the siblings through the handle (`eachRow: [4, 1]`)

In [ ]:
%%js
const sizes = ['S', 'M', 'L', 'XL']
let size = 'M'
const mark = v => `${v === size ? '●' : '○'} ${v}`
const shirt = pi.dialog({
  title: 'T-shirt', body: 'one button per value', eachRow: [4, 1],
  ...Object.fromEntries(sizes.map(v => [mark(v), () => {
    size = v
    sizes.forEach((x, i) => shirt.relabel(i, mark(x)))
    return mark(v)
  }])),
  Done() {},
})
await shirt
size

### 1.6 Checkboxes (a button returns its new label)

In [ ]:
%%js
const on = { lint: true, tests: false, docs: false }
const box = k => `[${on[k] ? 'x' : ' '}] ${k}`
await pi.dialog({
  title: 'Before commit', eachRow: [3, 1],
  ...Object.fromEntries(Object.keys(on).map(k => [box(k), () => ((on[k] = !on[k]), box(k))])),
  Done() {},
})
on

### 1.7 A live body (`.text` from a timer)

In [ ]:
%%js
let n = 0
const live = pi.dialog({ title: 'Live', body: 'tick 0', Stop() {} })
const t = setInterval(() => live.text({ body: `tick ${++n}`, footer: new Date().toLocaleTimeString() }), 500)
await live
clearInterval(t)
n

### 1.8 Non-modal toast: top-right, keys stay with Claude Code, ←/→ picks, `or` is pressed after `timeout`

In [ ]:
%%js
await pi.dialog({ title: 'Saved', body: 'auto-answers in 4 s', timeout: 4000, or: 'Keep', Keep() {}, Undo() {} })

## 2. Dialogs together with the rest of the API

### 2.1 `/effort` as a radio row, read back from the status line (`pi`` + `tui.wait`)

In [ ]:
%%js
const levels = ['low', 'medium', 'high', 'xhigh', 'max']
const effortOf = text => text.match(/\b(low|medium|high|xhigh|max)\s*·\s*\/effort/)?.[1]
await pi.tui.wait(() => true, 500) // a fresh frame
let effort = effortOf(pi.tui.state.scr?.text ?? '')
const dot = v => `${v === effort ? '●' : '○'} ${v}`
const ef = pi.dialog({
  title: '/effort', body: 'the dot moves once the status line agrees', eachRow: [5],
  ...Object.fromEntries(levels.map(v => [dot(v), async () => {
    await pi`/effort ${v}`
    if (await pi.tui.wait(s => effortOf(s.text) === v, 2500)) {
      effort = v
      levels.forEach((x, i) => ef.relabel(i, dot(x)))
    }
    return dot(v)
  }])),
})
await ef
effort

### 2.2 What is on screen (`tui.wait` + `tui.state.scr`)

In [ ]:
%%js
await pi.tui.wait(() => true, 500)
const { rows, cursor } = pi.tui.state.scr
await pi.dialog({
  title: `screen ${pi.tui.state.cols}×${pi.tui.state.rows} · cursor ${cursor.row},${cursor.col}`,
  body: rows.filter(r => r.trim()).slice(-8).map(r => r.slice(0, 70)).join('\n'),
  OK() {},
})

### 2.3 Command output in a dialog (`pi.sh`)

In [ ]:
%%js
const log = await pi.sh`git -C ${process.cwd()} log --oneline -6`
await pi.dialog({ title: `git log (exit ${log.code})`, body: (log.stdout || log.stderr).trim(), OK() {} })

### 2.4 Menu → confirm → input box (`pi.menu` + `pi.BUFFER`)

In [ ]:
%%js
const prompt = await pi.menu(['explain this repo', 'list the TODOs', 'write tests for <file>'])
if (prompt && (await pi.dialog({ body: `Put "${prompt}" in the input box?`, Yes() {}, No() {} })) === 'Yes') pi.BUFFER = prompt
prompt

### 2.5 Tab title (`tui.title`: D-Bus tab in Yakuake / Konsole / Guake, window title elsewhere)

In [ ]:
%%js
await pi.dialog({
  title: 'Tab title', eachRow: [3],
  'Set ✻ demo': async () => (await pi.tui.title('✻ demo'), 'Set ✻ demo'),
  Restore: async () => (await pi.tui.title(null), 'Restore'),
  Done() {},
})

### 2.6 Toggle a `pi.sed` from a checkbox (`''` blanks a row, `null` leaves it)

In [ ]:
%%js
let hide = null
const hideLabel = () => `[${hide ? 'x' : ' '}] blank the status-line rows`
await pi.dialog({
  title: 'pi.sed', eachRow: 'vertical',
  [hideLabel()]: () => {
    if (hide) { hide(); hide = null } else hide = pi.sed(ln => (/mode on|· \/effort/.test(ln) ? '' : null))
    return hideLabel()
  },
  Done() {},
})
nb.push(() => hide?.())

### 2.7 A `/hello` command that answers with a toast (`pi.command`)

In [ ]:
%%js
nb.push(pi.command('hello', ([name = 'there']) => pi.dialog({ title: '/hello', body: `hello ${name}`, timeout: 3000 })))
'now type  /hello world  in Claude Code' 

### 2.8 Ctrl+G: a quick-actions grid (key hook; `false` skips the default, `tui.press`)

In [ ]:
%%js
nb.push(pi.on({
  CtrlG: () => {
    pi.dialog({
      title: 'Ctrl+G', eachRow: [3],
      hints: { Transcript: 'Ctrl+O', History: 'Ctrl+R', Copy: '/copy' },
      Transcript: () => pi.tui.press('ctrl-o').then(() => {}),
      History: () => pi.tui.press('ctrl-r').then(() => {}),
      Copy: () => pi`/copy`.then(() => {}),
    })
    return false // not Claude Code's Ctrl+G
  },
}))
'press Ctrl+G in the terminal' 

### 2.9 A toast after every turn (the Stop hook)

In [ ]:
%%js
nb.push(pi.on({
  stop: p => pi.dialog({
    title: 'Claude finished', body: (p?.last_assistant_message ?? '').split('\n')[0].slice(0, 70),
    timeout: 5000, or: 'OK', OK() {}, Copy: () => pi`/copy`.then(() => {}),
  }),
}))
'shows after the next reply' 

### 2.10 Port + URL dialog (`pi.ext.webpane`, published by the webpane.js plugin)

In [ ]:
%%js
await pi.ext.webpane?.boundPort(8080) // 8080 if free (start your server), null if taken

### 2.11 Desktop notification and focus (`pi.notify`, `tui.state.focused`)

In [ ]:
%%js
await pi.dialog({
  title: 'notify', body: `terminal focused: ${pi.tui.state.focused}`, eachRow: [2],
  'Send one': () => (pi.notify('hello from Jupyter'), 'Send one'),
  Done() {},
})

## 3. The rest of the API

### 3.1 State and settings (`tui.state`, `pi.arg`)

In [ ]:
%%js
pi.arg.fzf = "--pointer '▶' --height=50% --bind 'ctrl-d:become(dirname -- {})'"; // Tab completion's fzf
({ ...pi.tui.state, scr: pi.tui.state.scr && `${pi.tui.state.scr.rows.length} rows`, fzf: pi.arg.fzf })

### 3.2 Type into the input box (`pi.BUFFER`, write-only)

In [ ]:
%%js
pi.BUFFER = 'draft from Jupyter (not sent)' 

### 3.3 Run a line (`pi`` replaces the box and presses Enter; resolves once typed)

In [ ]:
%%js
await pi`/copy`
'copied the last reply' 

### 3.4 Keys (`tui.press`): open the transcript, then leave it

In [ ]:
%%js
await pi.tui.press('ctrl-o')
await new Promise(r => setTimeout(r, 1500))
await pi.tui.press('esc')

### 3.5 Wait for the screen (`tui.wait` with a predicate; false on timeout)

In [ ]:
%%js
await pi.tui.wait(s => /(auto|manual|plan) mode on|accept edits on/.test(s.text), 2000)

### 3.6 Tab completion source (`pi.complete`): type `#b` and press Tab

In [ ]:
%%js
nb.push(pi.complete('#', tok => ['#bug', '#feature', '#docs', '#refactor'].filter(t => t.startsWith(tok))))

### 3.7 Events (`pi.on`; `await pi.on.stop` waits for the next one)

In [ ]:
%%js
globalThis.focusLog = []
nb.push(pi.on({ focus: f => focusLog.push([new Date().toLocaleTimeString(), f]) }))
'click away from the terminal and back, then run the next cell' 

In [ ]:
%%js
focusLog

### 3.8 Clean up what these cells registered

In [ ]:
%%js
const n0 = nb.length
nb.splice(0).forEach(off => off?.())
await pi.tui.title(null);
`${n0} registrations removed` 